# JP Morgan Chase — AI Finance Agent
## Lab 01 : Data Engineering
**Author :** Fabrice William FOMHOM  
**Date :** March 2026  
**Objective :** Create and explore a realistic financial transactions dataset for fraud detection and portfolio analysis

## Business Context
As Data Engineers at JP Morgan Chase, our mission is to build a clean, 
reliable financial dataset that will power our AI Agent.

This dataset simulates real banking transactions including :
- Customer transactions (purchases, transfers, withdrawals)
- Fraud indicators
- Portfolio holdings
- Account balances

**This is the foundation of our entire AI Agent — bad data = bad agent.**

In [1]:
# ============================================================
# JP Morgan Chase — AI Finance Agent
# Lab 01 : Data Engineering
# Step 1 : Import libraries
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import random
import os

print("✅ All libraries imported successfully")
print(f"Pandas version : {pd.__version__}")
print(f"NumPy version  : {np.__version__}")

✅ All libraries imported successfully
Pandas version : 2.2.2
NumPy version  : 1.26.4


In [2]:
# ============================================================
# Step 2 : Define project paths
# ============================================================

# Base project directory
BASE_DIR = r"C:\Users\HP\jpmorgan_finance_agent"

# Subfolders
DATA_DIR    = os.path.join(BASE_DIR, "data")
OUTPUT_DIR  = os.path.join(BASE_DIR, "outputs")
SCRIPTS_DIR = os.path.join(BASE_DIR, "scripts")

# Confirm paths exist
for name, path in [("Data", DATA_DIR), ("Outputs", OUTPUT_DIR), ("Scripts", SCRIPTS_DIR)]:
    status = "✅ Found" if os.path.exists(path) else "❌ Missing"
    print(f"{status} — {name}: {path}")

✅ Found — Data: C:\Users\HP\jpmorgan_finance_agent\data
✅ Found — Outputs: C:\Users\HP\jpmorgan_finance_agent\outputs
✅ Found — Scripts: C:\Users\HP\jpmorgan_finance_agent\scripts


In [3]:
# ============================================================
# Step 3 : Generate JP Morgan Financial Dataset
# ============================================================

# Set seed for reproducibility (academic best practice)
np.random.seed(42)
random.seed(42)

# ── Parameters ──────────────────────────────────────────────
N_TRANSACTIONS = 1000          # number of transactions
START_DATE     = datetime(2024, 1, 1)
END_DATE       = datetime(2024, 12, 31)

# ── Reference data ──────────────────────────────────────────
CUSTOMERS = [f"CUST_{i:04d}" for i in range(1, 101)]   # 100 customers

CATEGORIES = [
    "groceries", "electronics", "travel", 
    "restaurant", "healthcare", "transfer", 
    "ATM_withdrawal", "online_shopping"
]

MERCHANTS = [
    "Amazon", "Walmart", "Delta Airlines", "Apple Store",
    "Whole Foods", "CVS Pharmacy", "Marriott Hotels",
    "Best Buy", "McDonald's", "Uber"
]

# ── Generate transactions ────────────────────────────────────
records = []

for i in range(N_TRANSACTIONS):
    
    # Random date between start and end
    days_gap = (END_DATE - START_DATE).days
    tx_date  = START_DATE + timedelta(days=random.randint(0, days_gap))
    
    # Random transaction amount (most small, few large)
    amount = round(np.random.exponential(scale=150), 2)
    amount = min(amount, 15000)   # cap at $15,000
    
    # Fraud logic : ~3% fraud rate (realistic for banking)
    is_fraud = 1 if random.random() < 0.03 else 0
    
    # Fraudulent transactions tend to be larger
    if is_fraud:
        amount = round(amount * random.uniform(3, 8), 2)
        amount = min(amount, 15000)
    
    records.append({
        "transaction_id" : f"TXN_{i+1:05d}",
        "date"           : tx_date.strftime("%Y-%m-%d"),
        "customer_id"    : random.choice(CUSTOMERS),
        "merchant"       : random.choice(MERCHANTS),
        "category"       : random.choice(CATEGORIES),
        "amount"         : amount,
        "is_fraud"       : is_fraud
    })

# ── Create DataFrame ─────────────────────────────────────────
df = pd.DataFrame(records)

print(f"✅ Dataset created : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Fraud cases     : {df['is_fraud'].sum()} ({df['is_fraud'].mean()*100:.1f}%)")
print(f"   Date range      : {df['date'].min()} → {df['date'].max()}")
print(f"   Total volume    : ${df['amount'].sum():,.2f}")

✅ Dataset created : 1000 rows × 7 columns
   Fraud cases     : 29 (2.9%)
   Date range      : 2024-01-03 → 2024-12-31
   Total volume    : $165,365.53


In [4]:
# ============================================================
# Step 4 : First look at the data
# ============================================================

# First 5 rows — always the first thing a Data Analyst does
print("=" * 60)
print("FIRST 5 TRANSACTIONS")
print("=" * 60)
print(df.head())

print("\n")

# Data types and structure
print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)
print(df.info())

FIRST 5 TRANSACTIONS
  transaction_id        date customer_id     merchant         category  \
0      TXN_00001  2024-11-23   CUST_0095  Whole Foods       restaurant   
1      TXN_00002  2024-04-24   CUST_0014   McDonald's      electronics   
2      TXN_00003  2024-10-29   CUST_0004      Walmart       restaurant   
3      TXN_00004  2024-04-29   CUST_0004   McDonald's       restaurant   
4      TXN_00005  2024-11-28   CUST_0054  Apple Store  online_shopping   

   amount  is_fraud  
0   70.39         0  
1  451.52         0  
2  197.51         0  
3  136.94         0  
4   25.44         0  


DATASET STRUCTURE
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  1000 non-null   object 
 1   date            1000 non-null   object 
 2   customer_id     1000 non-null   object 
 3   merchant        1000 non-null   object 
 4   ca

In [5]:
# ============================================================
# Step 5 : Data Cleaning & Save
# ============================================================

# Convert date column to proper datetime format
df["date"] = pd.to_datetime(df["date"])

# Add useful columns for analysis
df["month"]      = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["day_of_week"]= df["date"].dt.day_name()
df["quarter"]    = df["date"].dt.quarter

# Verify the fix
print("=" * 60)
print("AFTER CLEANING")
print("=" * 60)
print(df.dtypes)
print(f"\n✅ Date column is now : {df['date'].dtype}")
print(f"✅ Dataset shape      : {df.shape}")

# Save to CSV in our data/ folder
output_path = os.path.join(DATA_DIR, "jpmorgan_transactions.csv")
df.to_csv(output_path, index=False)
print(f"\n✅ Dataset saved to   : {output_path}")

AFTER CLEANING
transaction_id            object
date              datetime64[ns]
customer_id               object
merchant                  object
category                  object
amount                   float64
is_fraud                   int64
month                      int32
month_name                object
day_of_week               object
quarter                    int32
dtype: object

✅ Date column is now : datetime64[ns]
✅ Dataset shape      : (1000, 11)

✅ Dataset saved to   : C:\Users\HP\jpmorgan_finance_agent\data\jpmorgan_transactions.csv


In [6]:
# ============================================================
# Step 6 : Summary Statistics — Executive Report
# ============================================================

print("=" * 60)
print("JP MORGAN — TRANSACTIONS SUMMARY REPORT")
print("=" * 60)

print(f"\n📊 VOLUME")
print(f"   Total transactions : {len(df):,}")
print(f"   Total amount       : ${df['amount'].sum():,.2f}")
print(f"   Average transaction: ${df['amount'].mean():,.2f}")
print(f"   Largest transaction: ${df['amount'].max():,.2f}")

print(f"\n🚨 FRAUD")
print(f"   Fraud cases        : {df['is_fraud'].sum()}")
print(f"   Fraud rate         : {df['is_fraud'].mean()*100:.2f}%")
print(f"   Fraud amount       : ${df[df['is_fraud']==1]['amount'].sum():,.2f}")

print(f"\n👥 CUSTOMERS")
print(f"   Unique customers   : {df['customer_id'].nunique()}")
print(f"   Top merchant       : {df['merchant'].value_counts().index[0]}")
print(f"   Top category       : {df['category'].value_counts().index[0]}")

print(f"\n📅 TIME")
print(f"   Date range         : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"   Busiest month      : {df['month_name'].value_counts().index[0]}")
print(f"   Busiest day        : {df['day_of_week'].value_counts().index[0]}")

print("\n" + "=" * 60)
print("✅ Lab 01 Complete — Data Engineering")
print("=" * 60)

JP MORGAN — TRANSACTIONS SUMMARY REPORT

📊 VOLUME
   Total transactions : 1,000
   Total amount       : $165,365.53
   Average transaction: $165.37
   Largest transaction: $3,882.53

🚨 FRAUD
   Fraud cases        : 29
   Fraud rate         : 2.90%
   Fraud amount       : $23,736.73

👥 CUSTOMERS
   Unique customers   : 100
   Top merchant       : McDonald's
   Top category       : electronics

📅 TIME
   Date range         : 2024-01-03 → 2024-12-31
   Busiest month      : Dec
   Busiest day        : Saturday

✅ Lab 01 Complete — Data Engineering
